# FAZ A2R2 -- `n`-bagimsiz encoding ailesinde EC'ye ulasilabilir mi?## Neden bu defter**Faz A2** (`norm='none'`): A0 fidelity kernel ustel konsantre oluyor (egim **-1.400**,`R^2=0.9999`, dAIC **-55.7**), agirlik-kesikli GFQK olmuyor (-0.047 / -0.002, GA'larsifiri iceriyor). Guclu sonuc.**Faz A2R** (konfound testi): `norm='sqrt'` ile faz sacilimi `n`'den bagimsiz halegetirilince **A0 da ustel konsantre olmayi birakiyor** (egim -0.364, `R^2_pol 0.969 >R^2_ust 0.860`, dAIC **+13.5**). `Var` `n~12`'den sonra yaylaya oturuyor (toplam dusussadece 410x, `none`'da 5.4e9x) ve Meyer-Wallach **0.76**'da doyuyor -- Haar'a hicyaklasmiyor.Yani `sqrt` normda **EC YOK** -> oran testi iki konsantre-olmayan cekirdegikarsilastiriyordu, **vakum**. (`a0_marginal` uyarisi tam bunun icin yazilmisti.)## Bu defterin sordugu tek soru`sqrt` normda `bw=0.3` devreyi yeterince karistirmiyor. **`bw` yukseltilirse`n`-bagimsiz bir ailede de EC'ye ulasilir mi?**- **Ulasilirsa** -> A2 iddiasi "standart ZZ ailesinde" kosullu olmaktan cikar, GENEL olur.- **Ulasilamazsa** -> iddia kosullu kalir; bu da dururust bir sonuctur ve zaten  A2R'nin buldugu "n-bagimliligi EC'nin KAYNAGI" tezini guclendirir.## ON-KAYITLI KILIT KURALI  (sonuclar gorulmeden sabitlenmistir)**Asama 1.** `sqrt` normda, REF kosesinde (`depth=8, ent=1.0`), `bw` grid'i uzerindeSADECE A0 kosulur (Pauli havuzu gerekmez -> saniyeler). Her `bw` icin Katman B uydurulur.```KILIT_BW = min{ bw :  exponential == True                  VE  r2_exp   > 0.99      (ustel uyum kaliteli)                  VE  dAIC     < -10       (ustel, polinomu KESIN yeniyor)                  VE  MW(n_max) > 0.90     (fizik: Haar rejimine girmis) }```Dort sart **birden**. `r2_exp` tek basina YETMEZ -- ic ice modelleri ayirt etmiyor:duman testinde `bw=0.3`, `r2_exp=0.9954` ile gecmisti ama `dAIC=-0.2` (polinomla basa bas)ve `MW=0.757` (Haar'a hic yaklasmamis). A2R zaten o hucreyi gercek 9-noktali veride`dAIC=+13.5` ile **USTEL DEGIL** bulmustu. Kural bu mantik hatasi giderilerek sabitlendi.Hicbiri saglamazsa -> `KILIT_BW = None`, Asama 2 KOSULMAZ, sonuc"sqrt ailesinde bw<=3.0'da EC'ye ulasilamadi" diye raporlanir.**Asama 2.** `KILIT_BW`'de GFQK serileri kosulur ve **A2R ile AYNI oran kurali**uygulanir (yeni esik uydurulmaz):```oran = egim_GFQK / egim_A0AYAKTA  : oran < 0.50  VE  GA'lar ayrikCOKTU   : oran > 0.80  VEYA GA'lar ortusuyorBELIRSIZ: arasi```> **Not.** Cok buyuk `bw`'de devre `x`'ten bagimsiz rastgeleye yaklasabilir: EC'nin en> siddetli hali, ama gorev de ogrenilemez olur. Bunu Katman D (Faz B) yakalar; burada> `MW` ve `offdiag_mean` raporlanarak izlenir.

## Bolum 0 - Kurulum

In [ ]:
import os, json, time
from math import comb
import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy as cp
    xp, GPU = cp, True
    print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode(),
          f"{cp.cuda.Device(0).mem_info[1]/2**30:.1f} GB")
except Exception as e:
    xp, GPU = np, False
    print('GPU YOK, CPU ile devam (cok yavas olacak):', e)

CDT = xp.complex64 if GPU else xp.complex128
FDT = xp.float32   if GPU else xp.float64
to_xp = lambda a, dtype=None: xp.asarray(a, dtype=dtype)
to_np = lambda a: cp.asnumpy(a) if GPU and isinstance(a, cp.ndarray) else np.asarray(a)
def free_mem():
    if GPU: cp.get_default_memory_pool().free_all_blocks()

## Bolum 1 - SabitlerYeni olan tek sey `BW_GRID` ve kilit kurali. Digerleri A2/A2R ile AYNI.

In [ ]:
SHOT_BUDGETS = [1e3, 1e5, 1e7]
LOCK_BUDGET  = 1e5        # A2/A2R ile AYNI
MEAN_GATE    = 0.5
EFFRANK_GATE = 0.90
SEED         = 0

NORM     = 'sqrt'                          # bu defter SADECE n-bagimsiz aileyi inceler
BW_GRID  = [0.3, 0.6, 1.0, 1.5, 2.0, 3.0]  # 0.3 = A2R'nin kullandigi (EC yok cikmisti)
# --- ON-KAYITLI KILIT ESIKLERI ---
# Dort sart BIRDEN saglanmali. `r2_exp` TEK BASINA YETMEZ: ic ice modelleri ayirt
# etmiyor (duman testinde bw=0.3 r2_exp=0.9954 ile gecti ama dAIC=-0.2, yani polinomla
# basa bas; gercek 9-noktali koseda A2R o hucreyi dAIC=+13.5 ile USTEL DEGIL bulmustu).
R2_LOCK   = 0.99      # ustel uyumun kalitesi
DAIC_LOCK = -10.0     # ustel, polinomu KESIN yenmeli (dAIC bundan kucuk)
MW_LOCK   = 0.90      # fizik tabanli bagimsiz kontrol: Haar rejimine girmis mi
RATIO_HOLD, RATIO_BREAK = 0.50, 0.80       # A2R ile AYNI, yeniden uydurulmadi

SERIES  = {2: [4,6,8,10,12,14,16,18], 3: [4,6,8,10,12,14,16]}
N_FID   = [4,6,8,10,12,14,16,18,20]
DEPTHS  = [2, 8]
ENTS    = [0.0, 0.5, 1.0]
REF     = dict(depth=8, ent=1.0)

def M_of(n):
    return 400 if n <= 12 else (250 if n <= 16 else 200)

CKPT = 'phase_a2r2_ckpt'; os.makedirs(CKPT, exist_ok=True)
print(f'Asama 1: {len(BW_GRID)} bw x {len(N_FID)} n, SADECE A0 (ucuz)')
print(f'Asama 2: kilitlenen bw"de GFQK  w<=2 (n<=18), w<=3 (n<=16)')

## Bolum 2 - Cekirdek  (kaynak: `src/qk_core.py`, TEK KAYNAK)

In [ ]:
"""Kuantum kernel cekirdegi -- TEK KAYNAK.

Bu dosya defterlere OLDUGU GIBI GOMULUR (build_*_nb.py `open(...).read()` ile alir),
boylece Colab'de self-contained kalir ama kod tek yerde durur.

BEKLENEN ORTAM: `xp`, `GPU`, `CDT`, `FDT`, `to_xp`, `to_np` defterin 0. hucresinde
tanimlanmis olmali (cupy/numpy dagitimi).

Dogrulanmis (bkz. defterlerin Bolum 4):
  fwht == H^{otimes n} · pauli_expect_fast == naif yol (2.2e-16) · Parseval ·
  GFQK_{w<=n} == fidelity kernel · dort eksende S_req monoton · Katman B/C ayrimlari
"""


# ============================================================================
# DEVRE -- yogun 2^n x 2^n matris KURULMAZ
# ============================================================================

def fwht(a, normalize=True):
    """Son eksende H^{otimes n}. a: (..., 2^n). O(N n 2^n), yogun matris yok."""
    dim = a.shape[-1]
    n = int(np.log2(dim))
    lead = a.shape[:-1]
    b = a.reshape(*lead, *([2] * n))
    for ax in range(len(lead), len(lead) + n):
        x = xp.moveaxis(b, ax, -1)
        b = xp.moveaxis(xp.stack([x[..., 0] + x[..., 1], x[..., 0] - x[..., 1]], -1), -1, ax)
    b = b.reshape(*lead, dim)
    return b * (2.0 ** (-n / 2)) if normalize else b


def sign_tables(n):
    z = np.arange(2 ** n)
    S = 1.0 - 2.0 * ((z[:, None] >> np.arange(n - 1, -1, -1)[None, :]) & 1)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    SP = np.stack([S[:, i] * S[:, j] for i, j in pairs], 1) if pairs else np.zeros((2 ** n, 0))
    return S, SP, pairs


def encode(X, n, depth, ent, bw, norm='none'):
    """ZZ-feature map, `depth` tekrar. Doner: (M, 2^n) statevector.

    faz(z) = -( bw * sum_i x_i s_i(z)  +  ent * bw^2 * sum_{i<j} x_i x_j s_ij(z) )

    `norm` -- `n`-BAGIMLI KONFOUND kontrolu:
      'none' : ham. Terim sayilari n ve n(n-1)/2 oldugu icin faz sacilimi
               sirasiyla ~sqrt(n) ve ~n buyur -> BUYUK n'de devre DAHA COK karisir.
               Bu, "n buyudu" ile "encoding guclendi" etkilerini karistirir.
      'sqrt' : her iki terim kendi terim sayisinin karekokune bolunur -> faz
               sacilimi n'den BAGIMSIZ. "Ayni harita, farkli n" yorumu gecerli olur.

    ent=0 -> faz tamamen tek-qubit -> durum CARPIM kalir (Meyer-Wallach tam 0),
    `norm` bunu degistirmez (tek-qubit terim carpimsal yapiyi bozmaz).
    """
    S, SP, pairs = sign_tables(n)
    Xp = np.stack([X[:, i] * X[:, j] for i, j in pairs], 1) if pairs else np.zeros((len(X), 0))
    c1, c2 = 1.0, 1.0
    if norm == 'sqrt':
        c1 = 1.0 / np.sqrt(n)
        c2 = 1.0 / np.sqrt(max(len(pairs), 1))
    elif norm != 'none':
        raise ValueError(norm)
    phase = -(c1 * bw * (to_xp(X, FDT) @ to_xp(S, FDT).T)
              + c2 * ent * (bw ** 2) * (to_xp(Xp, FDT) @ to_xp(SP, FDT).T))
    D = xp.exp(1j * phase).astype(CDT)
    psi = xp.zeros((len(X), 2 ** n), dtype=CDT)
    psi[:, 0] = 1.0
    for _ in range(depth):
        psi = fwht(psi) * D
    return psi


def meyer_wallach(psi, n):
    """Q = 2(1 - (1/n) sum_k tr(rho_k^2)).  0 = carpim, 1 = maksimal dolanik."""
    M = psi.shape[0]
    tot = xp.zeros(M, dtype=FDT)
    for k in range(n):
        a = xp.moveaxis(psi.reshape(M, *([2] * n)), k + 1, 1).reshape(M, 2, -1)
        rho = xp.einsum('mir,mjr->mij', a, a.conj())
        tot += xp.real(xp.einsum('mij,mji->m', rho, rho)).astype(FDT)
    return float(to_np(xp.mean(2.0 * (1.0 - tot / n))))


# ============================================================================
# PAULI DOMENI -- flip-maskesi + Walsh-Hadamard hizli yolu
# ============================================================================

def _popcount(v):
    v = np.asarray(v, dtype=np.int64).copy()
    r = np.zeros_like(v)
    while np.any(v):
        r += v & 1
        v >>= 1
    return r


def _spread4(v, n):
    v = np.asarray(v, dtype=np.int64)
    out = np.zeros_like(v)
    for k in range(n):
        out += ((v >> k) & 1) * (4 ** k)
    return out


def pool_layout(n, h):
    """Doner: (flip_maskeleri, [(g_dizisi, pauli_indeksleri) her f icin],
               tum_indeksler_sirali, tum_agirliklar_sirali).  Havuz: agirlik <= h."""
    allz = np.arange(2 ** n)
    pc = _popcount(allz)
    fs = np.where(pc <= h)[0]
    blocks, idx_all, w_all = [], [], []
    for f in fs:
        g = np.where(pc[allz | f] <= h)[0]            # agirlik = |f OR g|
        pidx = 2 * _spread4(f & g, n) + _spread4(f & ~g, n) + 3 * _spread4(g & ~f, n)
        blocks.append((g, pidx))
        idx_all.append(pidx)
        w_all.append(pc[g | f])
    idx_all = np.concatenate(idx_all)
    w_all = np.concatenate(w_all)
    o = np.argsort(idx_all)
    return fs, blocks, idx_all[o], w_all[o]


def pauli_expect_fast(psi, n, layout):
    """<P_{f,g}> = Re[ (-i)^{|f&g|} * WHT_z( conj(psi_z) psi_{z^f} )[g] ]

    Ayni f'yi paylasan tum operatorler tek gather + tek WHT ile hesaplanir.
    n=12, w<=4: 46 666 operator yerine 794 donusum.
    """
    fs, blocks, _, _ = layout
    z = xp.arange(2 ** n)
    conj = psi.conj()
    cols, keys = [], []
    for f, (g, pidx) in zip(fs, blocks):
        Wt = fwht(conj * psi[:, z ^ int(f)], normalize=False)
        ph = ((-1j) ** to_xp(_popcount(np.bitwise_and(int(f), g)))).astype(CDT)
        cols.append(xp.real(Wt[:, to_xp(g.astype(np.int64))] * ph[None, :]).astype(FDT))
        keys.append(pidx)
    E = xp.concatenate(cols, axis=1)
    return E[:, to_xp(np.argsort(np.concatenate(keys)))]


def pauli_expect_naive(psi, n, idx):
    """Referans yol -- SADECE dogrulama icin (yavas)."""
    flip = np.zeros(len(idx), np.int64)
    phm = np.zeros(len(idx), np.int64)
    nY = np.zeros(len(idx), np.int64)
    for k in range(n):
        d = (np.asarray(idx) // (4 ** k)) % 4
        flip |= np.where((d == 1) | (d == 2), 1 << k, 0)
        phm |= np.where((d == 2) | (d == 3), 1 << k, 0)
        nY += (d == 2)
    z = np.arange(2 ** n, dtype=np.int64)
    P = to_np(psi)
    out = np.zeros((P.shape[0], len(idx)))
    for t in range(len(idx)):
        zf = z ^ flip[t]
        sgn = 1.0 - 2.0 * (_popcount(zf & phm[t]) & 1)
        out[:, t] = np.real(np.sum(P.conj() * sgn[None, :] * P[:, zf], 1) * (1j) ** int(nY[t]))
    return out


# ============================================================================
# KERNEL -- birim kosegene normalize + tahmin edicinin shot varyansi
# ============================================================================

def kernel_gfqk(E_pool, w_pool, H, p):
    """GFQK_{w<=H}, gurultulu, birim kosegene normalize. Doner (K~, sigma2_shot).

    Gurultu: qubit basina depolarizing, Pauli-transfer ozdegeri lam=1-4p/3 -> EGZAKT.
    Shot varyansi: classical shadow, operator basina 3^w.
      k = sum_i a_i b_i, Var(a^_i) ~ 3^{w_i}
      -> Var(k^) ~ 2 sum_i 3^{w_i} <psi~_i^2>,   <psi~_i^2> = mean(E[:,i]^2)
    `c`'nin BIRINCI kuvveti kullanilir; c^2 yazmak pay/paydada sadelesip gurultu
    eksenini ters cevirir (bkz. 02_PROJECT_STATUS.md hata #16).
    """
    w = np.asarray(w_pool)
    m = to_xp(w <= H)
    ww = to_xp(w[w <= H].astype(np.float64), FDT)
    E = E_pool[:, m] * xp.sqrt((1.0 - 4.0 * p / 3.0) ** (2 * ww))[None, :]
    K = E @ E.T
    dg = xp.maximum(xp.diag(K), 1e-30)
    s2 = float(to_np(2.0 * xp.sum((3.0 ** ww) * xp.mean(E ** 2, 0)) / xp.mean(dg) ** 2))
    return K / xp.sqrt(xp.outer(dg, dg)), s2


def kernel_fidelity(psi):
    """A0 referansi. diag zaten 1; tahmin edici compute-uncompute BINOM -> sigma2 None.
    Bu, GFQK'dan FARKLI bir olcum protokoludur; mutlak shot sayilari ayni para birimi
    degil (bkz. hata #17)."""
    return xp.abs(psi.conj() @ psi.T) ** 2, None


# ============================================================================
# TESHIS -- Katman A-shot / A-learn / B / C
# ============================================================================

def effrank(ev):
    ev = xp.clip(ev.real, 0, None)
    s1, s2 = ev.sum(), (ev ** 2).sum()
    return float(to_np(s1 * s1 / s2)) if float(to_np(s2)) > 0 else 0.0


def diagnose(K, sigma2_shot=None, budgets=(1e3, 1e5, 1e7)):
    """S_req = sigma2_shot / Var[k~]  --  `1/Var` DEGIL (hata #15)."""
    M = K.shape[0]
    off = K[~xp.eye(M, dtype=bool)]
    var = float(to_np(off.var()))
    mu = float(to_np(off.mean()))
    kc = xp.clip(off, 0.0, 1.0)
    s2 = float(to_np(xp.mean(kc * (1 - kc)))) if sigma2_shot is None else float(sigma2_shot)
    er = effrank(xp.linalg.eigvalsh(K))
    d = dict(offdiag_var=var, offdiag_mean=mu, sigma2_shot=s2, M=M,
             s_req=(s2 / var) if var > 0 else float('inf'),
             effrank_K=er, effrank_frac=er / M)
    for S in budgets:
        d[f'resolv@{S:.0e}'] = float(to_np(xp.mean(xp.abs(off - mu) > 3 * np.sqrt(s2 / S))))
    return d


def layers_ac(d, budget, mean_gate, effrank_gate):
    """Doner (durum, a_shot, a_learn). Katman C ONCE: K->11^T ise bandwidth cokusu, EC DEGIL.

    UYARI: a_learn (effrank/M -> 1) EC'ye OZGU DEGILDIR -- carpim durumlarinda da
    tetiklenir ve n>=14'te 1.0000'da doyar. Bolge listesi Katman B (ustel onayi) ile
    KAPILANMALIDIR (bkz. 02_PROJECT_STATUS.md 6.96).
    """
    if d['offdiag_mean'] >= mean_gate:
        return 'COLLAPSE', False, False
    a_shot = d['s_req'] > budget
    a_learn = d['effrank_frac'] > effrank_gate
    if a_shot and a_learn:
        return 'EC-BOTH', True, True
    if a_shot:
        return 'EC-SHOT', True, False
    if a_learn:
        return 'EC-LEARN', False, True
    return 'NO-EC', False, False


def layer_b(ns, vals, n_boot=2000, seed=0):
    """log V ~ a + b*n (ustel)  vs  log V ~ a' + b'*log n (polinom), AIC ile.
    `exponential` = egim GA ust siniri < 0  VE  AIC ustelde daha kucuk.

    UYARI: `exponential=False`, "EC yok" DEMEK DEGILDIR -- egri henuz bukulme
    bolgesinde olabilir (asimptota girmemis). Bkz. 6.96 acik #1.
    """
    ns = np.asarray(ns, float)
    v = np.log(np.maximum(np.asarray(vals, float), 1e-300))
    k = len(ns)
    if k < 3:
        return None

    def fit(x, y):
        b, a = np.polyfit(x, y, 1)
        r = y - (a + b * x)
        ss = ((y - y.mean()) ** 2).sum()
        return b, (1 - (r ** 2).sum() / ss if ss > 0 else 0.0), (r ** 2).sum()

    b_e, r2_e, ss_e = fit(ns, v)
    b_p, r2_p, ss_p = fit(np.log(ns), v)
    aic = lambda ss: k * np.log(max(ss, 1e-300) / k) + 4
    rng = np.random.default_rng(seed)
    bs = []
    while len(bs) < n_boot:                       # dejenere cekilis (tek n) egimi bozar
        i = rng.integers(0, k, k)
        if len(np.unique(ns[i])) >= 2:
            bs.append(np.polyfit(ns[i], v[i], 1)[0])
    bs = np.array(bs)
    return dict(slope=float(b_e), slope_lo=float(np.percentile(bs, 2.5)),
                slope_hi=float(np.percentile(bs, 97.5)), r2_exp=float(r2_e),
                r2_poly=float(r2_p), d_aic=float(aic(ss_e) - aic(ss_p)),
                exponential=bool(np.percentile(bs, 97.5) < 0 and aic(ss_e) < aic(ss_p)))


def enc(X, n, depth, ent, bw):
    return encode(X, n, depth, ent, bw=bw, norm=NORM)

def diag_(K, s2=None):
    return diagnose(K, s2, budgets=tuple(SHOT_BUDGETS))

def layers_(d):
    return layers_ac(d, LOCK_BUDGET, MEAN_GATE, EFFRANK_GATE)

## Bolum 3 - DOGRULAMA (gecmezse ASAGI DEVAM ETME)`[4]` bu defterin en kritik testi: **`bw` buyurken faz sacilimi `n`'den bagimsiz KALIYOR mu?**Kalmiyorsa normalizasyon yuksek `bw`'de bozuluyordur ve tum test gecersizdir.

In [ ]:
def _validate():
    rng = np.random.default_rng(0); n = 4
    X = rng.uniform(0, 2*np.pi, size=(24, n))
    Hm = np.array([[1,1],[1,-1]], float)/np.sqrt(2); Hn = Hm
    for _ in range(n-1): Hn = np.kron(Hn, Hm)
    v = rng.normal(size=(5, 2**n))+1j*rng.normal(size=(5, 2**n))
    assert np.abs(to_np(fwht(to_xp(v, CDT))) - v @ Hn.T).max() < 1e-5
    print('[1a] OK  FWHT == H^{otimes n}')

    psi = enc(X, n, 2, 1.0, 0.3); lay = pool_layout(n, n)
    err = np.abs(to_np(pauli_expect_fast(psi, n, lay)) - pauli_expect_naive(psi, n, lay[2])).max()
    assert err < 1e-4
    print(f'[1b] OK  hizli Pauli yolu == naif yol (maxerr {err:.2e})')

    for bw in BW_GRID:
        for d in DEPTHS:
            mwz = meyer_wallach(enc(X, n, d, 0.0, bw), n)
            assert mwz < 1e-5, f'bw={bw} depth={d} ent=0 dolanik! MW={mwz:.2e}'
    print('[2] OK  her bw"de ent=0 -> tam carpim durumu (MW=0)')

    E = pauli_expect_fast(psi, n, lay)
    assert np.abs(to_np(xp.sum(E**2, 1))/2**n - 1).max() < 1e-4
    Kp, _ = kernel_gfqk(E, lay[3], n, 0.0); Kf, _ = kernel_fidelity(psi)
    assert np.abs(to_np(Kp-Kf)).max() < 1e-4
    print('[3] OK  Parseval + GFQK_{w<=n} == fidelity kernel')

    # [4] EN KRITIK: normalizasyon yuksek bw"de de n-bagimsizligi koruyor mu?
    print('[4] faz standart sapmasi vs n (ent=1, depth=1) -- her bw icin:')
    for bw in BW_GRID:
        sds = []
        for nn in [4, 8, 12, 16, 20]:
            Xn = np.random.default_rng(5).uniform(0, 2*np.pi, size=(48, nn))
            S, SP, pairs = sign_tables(nn)
            Xp = np.stack([Xn[:, i]*Xn[:, j] for i, j in pairs], 1)
            c1, c2 = 1/np.sqrt(nn), 1/np.sqrt(len(pairs))
            sds.append(float(np.std(-(c1*bw*(Xn @ S.T) + c2*(bw**2)*(Xp @ SP.T)))))
        ratio = max(sds)/min(sds)
        flag = 'OK' if ratio < 1.5 else 'BOZUK'
        print(f'     bw={bw:<4} sd={[round(x,2) for x in sds]}  max/min={ratio:.2f}  {flag}')
        assert ratio < 1.5, f'bw={bw}: sqrt normalizasyonu n-bagimsizligi KORUMUYOR'
    print('     OK  normalizasyon tum bw grid"inde saglam -> test gecerli')

    assert abs(diag_(xp.eye(50, dtype=FDT))['effrank_frac'] - 1.0) < 1e-6
    assert abs(diag_(xp.ones((50, 50), dtype=FDT))['effrank_frac'] - 0.02) < 1e-6
    print('[5] OK  effrank_frac: K=I -> 1.000, K=11^T -> 1/M')

    nv = np.array([4,6,8,10,12], float)
    assert layer_b(nv, np.exp(-1.4*nv))['exponential'] is True
    assert layer_b(nv, nv**-3.0)['exponential'] is False
    print('[6] OK  Katman B ustel/polinom ayrimi')

    assert RATIO_HOLD == 0.50 and RATIO_BREAK == 0.80, 'oran esikleri A2R ile ayni olmali'
    print('[7] OK  oran esikleri A2R ile AYNI (yeniden uydurulmadi)')

    # [8] kilit kurali r2 ile dAIC"i AYIRT ediyor mu? (duman testi hatasi)
    nv2 = np.array([4,6,8,10,12,14,16,18,20], float)
    marj = layer_b(nv2, 1.8e-2*(nv2/4.0)**-2.5)          # polinom: r2_ust yuksek AMA dAIC>0
    kesin = layer_b(nv2, np.exp(-1.4*nv2))               # gercek ustel
    assert marj['r2_exp'] > 0.95 and marj['d_aic'] > DAIC_LOCK,         f'polinom egri dAIC kapisini gecti: {marj["d_aic"]:.1f}'
    assert kesin['d_aic'] < DAIC_LOCK, f'gercek ustel dAIC kapisinda kaldi: {kesin["d_aic"]:.1f}'
    print(f'[8] OK  dAIC kapisi ayirt ediyor: polinom dAIC={marj["d_aic"]:+.1f} '
          f'(r2_ust={marj["r2_exp"]:.3f}) vs ustel dAIC={kesin["d_aic"]:+.1f}')
    print('\nTUM DOGRULAMALAR GECTI')

_validate()

## Bolum 4 - ASAMA 1: `bw` supurmesi (SADECE A0) + kilit

In [ ]:
def ck(tag): return os.path.join(CKPT, f'{tag}.json')

def run_fid(bw):
    tag = f'fid_bw{bw}'.replace('.', 'p')
    if os.path.exists(ck(tag)):
        return {int(k): v for k, v in json.load(open(ck(tag))).items()}
    out = {}; t0 = time.time()
    for n in N_FID:
        M = M_of(n)
        X = np.random.default_rng(SEED+777*n).uniform(0, 2*np.pi, size=(M, n))
        res = {}
        for dep in DEPTHS:
            for e in ENTS:
                psi = enc(X, n, dep, e, bw)
                d = diag_(kernel_fidelity(psi)[0]); d['mw'] = meyer_wallach(psi, n)
                d['status'], d['a_shot'], d['a_learn'] = layers_(d)
                res[f'{dep}|{e}'] = d
                del psi; free_mem()
        out[n] = res
    json.dump({str(k): v for k, v in out.items()}, open(ck(tag), 'w'))
    print(f'  bw={bw:<4} ({time.time()-t0:5.1f}s)')
    return out

rk = f'{REF["depth"]}|{REF["ent"]}'
FID, SL0 = {}, {}
print('=== ASAMA 1: A0 fidelity, norm=sqrt, bw supurmesi ===')
for bw in BW_GRID:
    FID[bw] = run_fid(bw)
    SL0[bw] = layer_b(N_FID, [FID[bw][n][rk]['offdiag_var'] for n in N_FID])

print(f'\n{"bw":>5} {"egim":>8} {"GA":>20} {"R2ust":>7} {"R2pol":>7} {"dAIC":>7} '
      f'{"USTEL":>6} {"MW(n=20)":>9} {"Var(n=20)":>10} {"dusus":>9}')
for bw in BW_GRID:
    s = SL0[bw]; r20 = FID[bw][N_FID[-1]][rk]; r4 = FID[bw][N_FID[0]][rk]
    print(f'{bw:>5} {s["slope"]:>8.3f} [{s["slope_lo"]:>+7.3f},{s["slope_hi"]:>+7.3f}] '
          f'{s["r2_exp"]:>7.4f} {s["r2_poly"]:>7.4f} {s["d_aic"]:>7.1f} '
          f'{str(s["exponential"]):>6} {r20["mw"]:>9.3f} {r20["offdiag_var"]:>10.2e} '
          f'{r4["offdiag_var"]/r20["offdiag_var"]:>9.3g}x')

# --- ON-KAYITLI KILIT ---
def lock_ok(bw):
    s_ = SL0[bw]; mw = FID[bw][N_FID[-1]][rk]['mw']
    return (s_['exponential'] and s_['r2_exp'] > R2_LOCK
            and s_['d_aic'] < DAIC_LOCK and mw > MW_LOCK)

cand = [bw for bw in BW_GRID if lock_ok(bw)]
LOCK_BW = min(cand) if cand else None
print(f'\nKilit kurali (DORDU BIRDEN): exponential  VE  r2_exp>{R2_LOCK}  VE  '
      f'dAIC<{DAIC_LOCK}  VE  MW(n_max)>{MW_LOCK}')
for bw in BW_GRID:
    s_ = SL0[bw]; mw = FID[bw][N_FID[-1]][rk]['mw']
    chk = [('ust', s_['exponential']), (f'r2>{R2_LOCK}', s_['r2_exp'] > R2_LOCK),
           (f'dAIC<{DAIC_LOCK}', s_['d_aic'] < DAIC_LOCK), (f'MW>{MW_LOCK}', mw > MW_LOCK)]
    print(f'   bw={bw:<4} ' + '  '.join(f'{k}:{"OK" if v else "--"}' for k, v in chk)
          + ('   <== ADAY' if all(v for _, v in chk) else ''))
print(f'Aday bw"ler: {cand if cand else "YOK"}')
print(f'>>> KILIT_BW = {LOCK_BW}')
if LOCK_BW is None:
    print('\n*** sqrt ailesinde bw<=%.1f araliginda EC"ye ULASILAMADI.' % max(BW_GRID))
    print('    Asama 2 KOSULMAYACAK. A2 iddiasi "standart ZZ ailesinde" KOSULLU kalir.')
    print('    Bu, A2R"nin "n-bagimliligi EC"nin KAYNAGI" tezini guclendirir.')

## Bolum 5 - ASAMA 2: kilitlenen `bw`"de GFQK serileri

In [ ]:
SER, SLG = {}, {}
if LOCK_BW is not None:
    print(f'=== ASAMA 2: GFQK, norm=sqrt, bw={LOCK_BW} ===')
    for H in sorted(SERIES):
        tag = f'H{H}_bw{LOCK_BW}'.replace('.', 'p')
        if os.path.exists(ck(tag)):
            SER[H] = {int(k): v for k, v in json.load(open(ck(tag))).items()}
        else:
            out = {}
            for n in SERIES[H]:
                t0 = time.time(); M = M_of(n)
                X = np.random.default_rng(SEED+777*n).uniform(0, 2*np.pi, size=(M, n))
                lay = pool_layout(n, H); w = lay[3]; res = {}
                for dep in DEPTHS:
                    for e in ENTS:
                        psi = enc(X, n, dep, e, LOCK_BW)
                        mw = meyer_wallach(psi, n)
                        E = pauli_expect_fast(psi, n, lay); del psi; free_mem()
                        K, s2 = kernel_gfqk(E, w, H, 0.0)
                        d = diag_(K, s2); d['mw'] = mw
                        d['status'], d['a_shot'], d['a_learn'] = layers_(d)
                        res[f'{dep}|{e}'] = d
                        del K, E; free_mem()
                out[n] = res
                r = res[rk]
                print(f'  w<={H} n={n:2d} ({time.time()-t0:6.1f}s) Var={r["offdiag_var"]:.2e} '
                      f'S_req={r["s_req"]:.2e} er/M={r["effrank_frac"]:.3f} {r["status"]}')
            json.dump({str(k): v for k, v in out.items()}, open(ck(tag), 'w'))
            SER[H] = out
        SLG[H] = layer_b(SERIES[H], [SER[H][n][rk]['offdiag_var'] for n in SERIES[H]])
    print('\nASAMA 2 BITTI')
else:
    print('KILIT_BW yok -> Asama 2 atlandi.')

## Bolum 6 - KARAR (A2R ile AYNI oran kurali)

In [ ]:
def compare(sg, sa):
    ratio = sg['slope']/sa['slope'] if sa['slope'] != 0 else float('inf')
    disjoint = sg['slope_lo'] > sa['slope_hi']
    if ratio < RATIO_HOLD and disjoint: v = 'AYAKTA'
    elif ratio > RATIO_BREAK or not disjoint: v = 'COKTU'
    else: v = 'BELIRSIZ'
    return ratio, disjoint, v

print('='*74)
if LOCK_BW is None:
    print('SONUC: sqrt (n-bagimsiz) ailede EC"ye ulasilamadi -> A2 iddiasi KOSULLU kalir.')
    print('  Raporlanacak ifade: "Sabit terim-basi bandwidth"li standart ZZ feature map')
    print('  ailesinde fidelity kernel ustel konsantre olur, agirlik-kesikli olmaz.')
    print('  Encoding n-bagimsiz olacak sekilde yeniden olceklenirse fidelity kernel de')
    print('  ustel konsantre olmayi birakir -> n-bagimliligi EC"nin KAYNAGIDIR."')
    VERDICT = dict(lock_bw=None, general_claim=False)
else:
    sa = SL0[LOCK_BW]
    print(f'SONUC: sqrt ailede bw={LOCK_BW} ile EC"ye ULASILDI (A0 egim {sa["slope"]:+.3f}, '
          f'R2={sa["r2_exp"]:.4f})')
    print(f'\n{"seri":>7} | {"egim_GFQK":>10} {"egim_A0":>9} {"oran":>6} {"GA ayrik":>9} | karar')
    res = {}
    for H in sorted(SERIES):
        r, dj, v = compare(SLG[H], sa); res[H] = v
        print(f'  w<={H}   | {SLG[H]["slope"]:>10.3f} {sa["slope"]:>9.3f} {r:>6.3f} '
              f'{str(dj):>9} | {v}')
    allhold = all(v == 'AYAKTA' for v in res.values())
    print()
    if allhold:
        print('>>> IDDIA GENEL: n-bagimsiz encoding ailesinde de fidelity kernel ustel')
        print('    konsantre olurken agirlik-kesikli GFQK olmuyor. A2 sonucu artik')
        print('    "standart ZZ ailesi"ne KOSULLU DEGIL.')
    else:
        print('>>> IDDIA KOSULLU KALIR: n-bagimsiz ailede ayrim korunmuyor.')
        print('    Coken seriler:', [f'w<={H}' for H, v in res.items() if v == 'COKTU'])
    VERDICT = dict(lock_bw=LOCK_BW, general_claim=bool(allhold),
                   per_series={f'w{H}': v for H, v in res.items()},
                   ratios={f'w{H}': SLG[H]['slope']/sa['slope'] for H in sorted(SERIES)})
print('='*74)

print('\n--- referans: Faz A2 (norm=none) sonucu ---')
print('  A0 -1.4006 [-1.419,-1.388] R2u=0.9999 dAIC=-55.7 USTEL')
print('  w<=2 -0.047 (GA sifiri iceriyor)   w<=3 -0.002 (GA sifiri iceriyor)')

## Bolum 7 - Grafik + indirme

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': .3, 'figure.autolayout': True})
FIGS = []
ns = np.array(N_FID, float)

fig, axs = plt.subplots(1, 2, figsize=(9.6, 3.9))
for bw in BW_GRID:
    lab = f'bw={bw}' + ('  <-- KILIT' if bw == LOCK_BW else '')
    axs[0].semilogy(N_FID, [FID[bw][n][rk]['offdiag_var'] for n in N_FID], 'o-',
                    lw=2.2 if bw == LOCK_BW else 1.2, ms=4, label=lab)
    axs[1].plot(N_FID, [FID[bw][n][rk]['mw'] for n in N_FID], 'o-',
                lw=2.2 if bw == LOCK_BW else 1.2, ms=4, label=lab)
v0ref = FID[BW_GRID[0]][N_FID[0]][rk]['offdiag_var']
axs[0].semilogy(ns, v0ref*4.0**(-(ns-ns[0])), '--', c='gray', lw=1.1,
                label=r'Haar $4^{-n}$')
axs[0].set_xlabel('$n$'); axs[0].set_ylabel(r'Var$[k]$')
axs[0].set_title(r"A0, norm='sqrt': $bw$ supurmesi", fontsize=9)
axs[0].legend(fontsize=6.5)
axs[1].set_xlabel('$n$'); axs[1].set_ylabel('Meyer-Wallach'); axs[1].set_ylim(0, 1.02)
axs[1].axhline(1.0, color='gray', ls='--', lw=1)
axs[1].set_title('Dolaniklik doyumu (1.0 = maksimal)', fontsize=9)
fig.savefig('R2_bw_sweep.png', bbox_inches='tight'); FIGS.append('R2_bw_sweep.png')

if LOCK_BW is not None:
    fig2, ax = plt.subplots(figsize=(5.2, 3.9))
    ax.semilogy(N_FID, [FID[LOCK_BW][n][rk]['offdiag_var'] for n in N_FID], 'o-',
                c='crimson', lw=2, ms=5, label=f'A0 fidelity (bw={LOCK_BW})')
    v0 = FID[LOCK_BW][N_FID[0]][rk]['offdiag_var']
    ax.semilogy(ns, v0*4.0**(-(ns-ns[0])), '--', c='crimson', alpha=.45, lw=1.1,
                label=r'Haar $4^{-n}$')
    for H, c in zip(sorted(SERIES), ['#1f77b4', '#2ca02c']):
        ax.semilogy(SERIES[H], [SER[H][n][rk]['offdiag_var'] for n in SERIES[H]], 's-',
                    c=c, lw=2, ms=4, label=f'GFQK $w\\leq{H}$')
    ax.set_xlabel('$n$'); ax.set_ylabel(r'Var$[k]$')
    ax.set_title(f"$n$-bagimsiz ailede ayrim  (norm='sqrt', bw={LOCK_BW})", fontsize=9)
    ax.legend(fontsize=7.5)
    fig2.savefig('R3_locked_bw.png', bbox_inches='tight'); FIGS.append('R3_locked_bw.png')
plt.show()

json.dump(dict(verdict=VERDICT,
               A0_slopes_by_bw={str(bw): SL0[bw] for bw in BW_GRID},
               GFQK_slopes={f'w{H}': SLG[H] for H in SLG},
               params=dict(NORM=NORM, BW_GRID=BW_GRID, R2_LOCK=R2_LOCK,
                           RATIO_HOLD=RATIO_HOLD, RATIO_BREAK=RATIO_BREAK,
                           SERIES={str(k): v for k, v in SERIES.items()}, N_FID=N_FID,
                           DEPTHS=DEPTHS, ENTS=ENTS, REF=REF, SEED=SEED,
                           LOCK_BUDGET=LOCK_BUDGET, EFFRANK_GATE=EFFRANK_GATE)),
          open('phase_a2r2_results.json', 'w'), indent=1)
print('-> phase_a2r2_results.json')

import shutil
shutil.make_archive('phase_a2r2_ckpt', 'zip', CKPT)
try:
    from google.colab import files
    for f in ['phase_a2r2_ckpt.zip', 'phase_a2r2_results.json'] + FIGS:
        files.download(f)
except Exception as e:
    print('Colab disi, elle indir:', e)